# nb_export_release — 버전 고정 배포 내보내기 → PostgreSQL

Gold(Delta) → 피처/근거 표 → 검증 → 새 릴리스 적재 → 포인터 전환 (한 트랜잭션)

- 이 노트북에는 계산이 없습니다. `analytics/src`의 함수만 호출합니다.
- **빈 칸(`None`)은 Fabric/DevOps 담당과 확인해 채울 값입니다.** 비어 있으면 실행이 시작 전에 멈추고 빈 키를 알려 줍니다.
- 비밀번호·접속 문자열은 쓰지 않습니다. Key Vault 비밀 *이름*만 적습니다.
- PostgreSQL 표 구조는 **제안**입니다. market-state 서비스 담당과 합의 후 `release_export.py`의 DDL만 조정하세요.
- 처음에는 `dry_run=True`로 표 검사와 해시만 확인하세요(DB에 아무것도 만들지 않음).

In [ ]:
# === 설정 칸 (여기만 수정) ===
CODE_DIR = None
CFG = {
    'gold_kind': 'delta_path', 'gold_location': None, 'dataset_version': None, 'delta_version': None, 'strict': False,
    'pointer': 'current',           # 읽는 쪽이 보는 포인터 이름 (서비스 담당과 합의)
    'paramstyle': 'format', 'dialect': 'postgres',
    'dry_run': True,                # 처음엔 True. 확인 후 False
}
# PostgreSQL 접속: 값이 아니라 Key Vault 비밀 '이름'만 적습니다.
VAULT_URL = None                    # 예: 'https://<vault>.vault.azure.net/'
SECRET_NAMES = {'host': None, 'dbname': None, 'user': None, 'password': None}

In [ ]:
import sys, json
if CODE_DIR: sys.path.insert(0, CODE_DIR)
import run_release as RR
from fabric_io import get_secret

conn = None
if not CFG['dry_run']:
    import psycopg                                   # 환경에 없으면 설치 필요 (DevOps 확인)
    s = {k: get_secret(v, VAULT_URL) for k, v in SECRET_NAMES.items()}
    conn = psycopg.connect(host=s['host'], dbname=s['dbname'], user=s['user'], password=s['password'], sslmode='require')
try:
    result = RR.run_export(CFG, conn, spark=spark)
finally:
    if conn is not None: conn.close()

In [ ]:
print(json.dumps({k: result[k] for k in ('action','release_id','rows','content_sha256','pointer')}, indent=2))
print(json.dumps(result['gold'], ensure_ascii=False, indent=2, default=str))

**롤백**: `release_export.rollback_pointer(conn, '<이전 release_id>')`  — 데이터는 지우지 않고 포인터만 되돌립니다. 릴리스 목록은 `list_releases(conn)`.